---
title: "Ration Early or Run Dry?"
subtitle: "A reservoir trade-off in four TaqSim blocks"
author:
  - name: Dr. Tobias Siegfried
    affiliation: hydrosolutions GmbH
date: today
format:
  html:
    theme: cosmo
    toc: true
    toc-depth: 2
    number-sections: true
    code-fold: true
    code-summary: "Show the code"
    code-tools: true
    embed-resources: true
    include-in-header:
      - text: |
          <style>
          .cell-output-display img { max-width: 100%; height: auto; }
          main.content { max-width: 960px; }
          </style>
jupyter: python3
execute:
  echo: true
  warning: false
license: "CC BY 4.0"
---

# Вопрос

Водохранилище наполняется зимой. Летом из него берёт воду хозяйство. В большинстве лет воды хватает. В сухой год
её не хватает, и у оператора есть два способа управлять водохранилищем:

- **Подавать полностью, пока водохранилище не опустеет.** Хозяйство получает всё, что просит, так долго, как
  возможно. Если вода кончается, последние недели лета остаются почти ни с чем.
- **Лимитировать заранее.** Когда воды в водохранилище становится мало, подавать лишь часть того, что просит
  хозяйство. Воды хватает дольше, но дефицит случается чаще, в том числе в годы, когда воды и так хватило бы.

Что лучше? Это зависит от того, что считать. В этом блокноте собрана самая маленькая модель, которая это
показывает: четыре блока TaqSim, два рычага, две цели. Затем поиск поручается оптимизатору.

**Что вы вынесете из блокнота**

1. Как собрать и запустить небольшую водную систему в TaqSim, блок за блоком.
2. Почему две разумные цели могут тянуть в разные стороны и как увидеть все разумные компромиссы сразу.
3. Как поручить поиск оптимизатору и как читать то, что он возвращает.

> **Термины.** Инженеры называют «подавать полностью, пока не опустеет» *стандартной политикой управления*, а
> «лимитировать заранее» — *хеджированием*. План, который ни один другой план не превосходит по всем целям,
> называется *Парето-оптимальным*, а множество таких планов — *фронтом Парето*. В этом блокноте они называются
> «непревзойдённые планы».

# Система из четырёх блоков

TaqSim описывает водную систему как **сеть**: узлы, которые хранят, используют или передают воду, соединённые
рёбрами, по которым вода течёт. В нашей сети четыре узла и три ребра.

In [ ]:
# ── Setup: run this cell first. In Colab it takes about a minute. ──────────────────────────────────
# It fetches the course folder, installs TaqSim, and tells Python where the course code lives.
import os
import sys
import subprocess

if not os.path.exists("src/zarafshan_taqsim"):  # not inside the course folder yet (the normal case in Colab)
    if not os.path.exists("zarafshan-taqsim-course"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/hydrosolutions/zarafshan-taqsim-course.git"], check=True)
    os.chdir("zarafshan-taqsim-course")
try:
    import taqsim  # noqa: F401  (already installed when you work on your own computer)
except ImportError:
    %pip install -q deap ctrl-freak pymoo
    %pip install -q --no-deps "git+https://github.com/hydrosolutions/taqsim.git@v0.1.4"
sys.path[:0] = [os.path.abspath("src")]
print("Ready. Working folder:", os.getcwd())


In [ ]:
#| label: setup
# Standard tools
import inspect  # to show the source code of a function or class inside this notebook
import logging

import pandas as pd  # tables
from IPython.display import HTML, Markdown  # to show formatted text and HTML as cell output

# TaqSim itself: the four kinds of node we use, the edge that joins them, and the system that holds them
from taqsim import Demand, Edge, Sink, Source, Storage, TimeSeries, WaterSystem
from taqsim.node import NoLoss  # a loss rule that loses nothing (reservoirs need one)
from taqsim.node.events import DeficitRecorded  # the event a Demand node records when it is short
from taqsim.time import Frequency  # the length of one simulation step (monthly here)

# The helpers written for this coursebook (src/zarafshan_taqsim/rationing_coursebook.py)
from zarafshan_taqsim.rationing_coursebook import (
    CAPACITY_MM3,
    EVAPORATION_SHARE_EXAMPLE,
    MONTHLY_NEED_MM3,
    NO_RATIONING,
    PAPER_REGIME,
    START_STORAGE_MM3,
    YEARS,
    Plan,
    RationingRelease,
    SummerEvaporation,
    drought_window_start,
    explorer_html,
    explorer_payload,
    front_of,
    objectives,
    optimise_plans,
    plan_grid,
    plot_front_shift,
    plot_inflows,
    plot_network,
    plot_regime_comparison,
    plot_runs,
    plot_score_maps,
    plot_system_sketch,
    plot_tradeoff,
    score_plans,
    season_statistics,
    simulate_plan,
    synthetic_inflows,
)

logging.basicConfig(level=logging.WARNING)  # only show warnings and errors, not progress messages
pd.set_option("display.precision", 2)  # two decimals in tables

LANG = "ru"  # the web page shows all three languages and lets you switch; a notebook is built for one language


def say(en: str, uz: str, ru: str) -> Markdown:
    """Text in three languages: all three on the web page (the switcher picks one), one in a notebook."""
    if LANG:
        return Markdown({"en": en, "uz": uz, "ru": ru}[LANG])
    blocks = (("en", en), ("uz", uz), ("ru", ru))
    return Markdown("\n\n".join(f"::: {{.lang-{code}}}\n{text}\n:::" for code, text in blocks))


def show_source(thing) -> Markdown:
    """Show the source code of a function or class as a code block."""
    return Markdown(f"```python\n{inspect.getsource(thing)}```")

In [ ]:
#| label: fig-network
#| fig-cap: "The model as a network. Each node is one TaqSim block; each edge is named after the nodes it joins."
#| echo: false
_ = plot_network()

- **river** (река) — `Source`: каждый месяц подаёт воду в сеть по списку чисел.
- **reservoir** (водохранилище) — `Storage`: хранит воду до своей ёмкости, часть теряет (если мы так зададим) и
  часть сбрасывает по правилу. Здесь принимаются решения.
- **farm** (хозяйство) — `Demand`: каждый месяц запрашивает заданный объём и записывает, сколько не получило.
- **downstream** (нижний бьеф) — `Sink`: всё, что хозяйство не использовало, покидает сеть здесь.

In [ ]:
#| label: fig-system
#| fig-cap: "The same four blocks with their numbers. Water moves from left to right."
#| echo: false
_ = plot_system_sketch()

Числа взяты из классической статьи о надёжности водохранилищ (Hashimoto, Stedinger и Loucks, 1982, с. 17 и
таблица 1) с одним изменением. Река в статье многоводна зимой. Реки Центральной Азии питаются талыми водами снега и
ледников: пик стока приходится на лето, а зимой сток минимален. Поэтому мы меняем местами два сезона из статьи, а
всё остальное оставляем. [Раздел сравнения](#sec-paper-regime) прогоняет реку из статьи рядом с нашей. Объёмы в
миллионах кубометров (млн м³).

| Величина | Значение |
|---|---|
| Ёмкость водохранилища | 40 млн м³ |
| Потребность хозяйства летом | 45 млн м³ (7,5 млн м³ в месяц в течение шести месяцев) |
| Обязательный сброс зимой | 5 млн м³ (0,83 млн м³ в месяц в течение шести месяцев) |
| Приток зимой | в среднем 25 млн м³, стандартное отклонение 10 млн м³ (в статье: 40 и 15) |
| Приток летом | в среднем 40 млн м³, стандартное отклонение 15 млн м³ (в статье: 25 и 10) |

Летняя потребность (45) больше среднего летнего притока (40), поэтому хозяйство зависит от воды, накопленной
зимой. Водохранилище (40) меньше потребности, а наполнить его можно только из скудного зимнего стока, так что
запаса на ошибку почти нет.

Модельный год — шесть зимних месяцев, затем шесть летних; календарные даты не заданы. Читайте его как
гидрологический год, принятый в бассейне Зарафшана: зима — с октября по март, лето (вегетационный период) — с
апреля по сентябрь.

## Река

Ста лет измерений у нас нет, поэтому мы создаём реку сами: сто лет зимних и летних притоков со средними и
разбросом из таблицы выше. За сухим сезоном, как и в настоящих реках, чаще следует ещё один сухой сезон.

In [ ]:
#| label: river
inflows = synthetic_inflows()  # 100 years x 12 months = 1 200 numbers, each a month's inflow in Mm³

# Check the river against the numbers we asked for: averages, spread and the link from one season to the next
season_statistics(inflows).style.hide(axis="index").format(precision=2)

`synthetic_inflows()` всегда даёт одну и ту же реку (используется фиксированное зерно генератора случайных
чисел), поэтому любой результат в этом блокноте воспроизводим. Таблица сравнивает статистику реки с заданными
значениями; они совпадают с точностью до нескольких процентов.

In [ ]:
#| label: fig-river
#| fig-cap: "One hundred synthetic years. In many years the summer inflow is far below what the farm needs."
#| echo: false
_ = plot_inflows(inflows)

> **Чем это отличается от статьи.** (1) Сезоны поменяны местами, как объяснено выше. (2) Приток каждого сезона
> равномерно распределён по шести месячным шагам. (3) В статье правила управления выводятся математически; мы
> используем простое правило с двумя рычагами. (4) Связь между сезонами задана для логарифмов стока — это
> допущение. Поэтому числа в этом блокноте наши собственные, а не воспроизведение чисел статьи.

# Собираем в TaqSim

## Правило с двумя рычагами

Водохранилищу в TaqSim нужно *правило сброса*: небольшой класс, который говорит, сколько воды выпускать каждый
месяц. У нашего правила два рычага.

- **Порог** (trigger, млн м³): лимитирование начинается, когда в водохранилище меньше этого объёма.
- **Доля** (ration, от 0 до 1): какая часть летней потребности тогда подаётся.

In [ ]:
#| label: rule-source
#| echo: false
show_source(RationingRelease)

Читаем класс сверху вниз:

- `@dataclass(frozen=True)` делает его маленькой неизменяемой записью: в ней два рычага и больше ничего.
- `__params__` называет рычаги, которые может крутить оптимизатор, а `__bounds__` — в каких пределах.
- `trigger` и `ration` — сами рычаги со значениями по умолчанию (0 и 1: без лимитирования).
- `release(...)` TaqSim вызывает раз в месяц. Метод получает водохранилище (`node`, из которого читает текущий
  объём), приток этого месяца и шаг времени `t` и возвращает объём сброса в млн м³.

Порог 0 (или доля 1) означает полное отсутствие лимитирования.

## Соединяем четыре блока

Сборка модели TaqSim всегда состоит из одних и тех же четырёх шагов: создать пустую систему, добавить узлы,
добавить рёбра, проверить и запустить.

In [ ]:
#| label: wire
# 1. An empty system. One simulation step is one month.
system = WaterSystem(frequency=Frequency.MONTHLY)

# 2. The four nodes. Each has an id (its name in the network) and the settings its kind needs.
system.add_node(
    Source(
        id="river",
        inflow=TimeSeries(values=list(inflows)),  # one value per month, Mm³
    )
)
system.add_node(
    Storage(
        id="reservoir",
        capacity=CAPACITY_MM3,  # 40 Mm³; more than this spills
        initial_storage=START_STORAGE_MM3,  # 20 Mm³ in store when the simulation starts
        release_policy=RationingRelease(trigger=0.0, ration=1.0),  # our rule; these knobs mean no rationing
        loss_rule=NoLoss(),  # no evaporation for now
    )
)
system.add_node(
    Demand(
        id="farm",
        requirement=TimeSeries(values=list(MONTHLY_NEED_MM3) * YEARS),  # the 12 monthly needs, repeated 100 times
    )
)
system.add_node(Sink(id="downstream"))

# 3. The three edges. An edge is named after the two nodes it joins and carries whatever its source lets out.
for source, target in [("river", "reservoir"), ("reservoir", "farm"), ("farm", "downstream")]:
    system.add_edge(Edge(id=f"{source}_to_{target}", source=source, target=target))

# 4. Check that the network is complete (every node connected, every setting valid), then run all 1 200 months.
system.validate()
system.simulate(len(inflows))

Запуск системы ничего не печатает. Вместо этого каждый узел ведёт журнал **событий**: что он получил, сохранил,
сбросил, потерял или не смог подать, месяц за месяцем. Хозяйство записывает событие `DeficitRecorded` в каждом
месяце, когда получает меньше, чем нужно, и каждое такое событие хранит недостающий объём:

In [ ]:
#| label: first-read
farm = system.nodes["farm"]  # look up a node by its id

# Add up the missing volume over all deficit events, then divide by the number of years
not_delivered = sum(event.deficit for event in farm.events_of_type(DeficitRecorded))
print(f"Water not delivered: {not_delivered / YEARS:.2f} Mm³ per year, on average")

Далее `simulate_plan(inflows, plan)` выполняет ровно эти шаги (собрать, проверить, рассчитать, прочитать события)
и возвращает результат в удобном виде: объём в водохранилище по месяцам, дефицит по месяцам и три показателя,
которые вводятся ниже.

# Первый расчёт: подавать полностью, пока не опустеет

In [ ]:
#| label: fig-first-run
#| fig-cap: "No rationing. Left: water in the reservoir over twenty of the hundred years. Right: for each year, how short the worst month was."
no_rationing = simulate_plan(inflows, NO_RATIONING)  # NO_RATIONING is Plan(trigger=0, ration=1)
first_year = drought_window_start(no_rationing)  # a 20-year window that contains the worst drought

# One row per plan; the left panel shows the 20 years starting in first_year
_ = plot_runs({"Deliver in full until empty": no_rationing}, first_year=first_year)

На левой панели водохранилище наполняется каждую зиму (восходящие линии) и срабатывается каждое лето. В самые сухие
годы оно опустошается до нуля. Правая панель показывает худший месяц каждого года: в большинстве лет столбика нет
вовсе.

План оцениваем тремя числами:

- **Средний дефицит**: вода, которая была нужна хозяйству, но не подана, в млн м³ в год.
- **Худший месяц**: доля потребности, не поданная хозяйству в самом худшем месяце за сто лет.
- **Дефицитные годы**: число лет хотя бы с одним дефицитным месяцем.

In [ ]:
#| label: first-scores
#| echo: false
s = no_rationing.scores
say(
    f"Without rationing the farm is short in **{s.years_short} of 100 years**. On average **{s.mean_shortage:.1f} Mm³ "
    f"per year** is not delivered, which is little. But when it goes wrong, it goes badly wrong: in the worst month "
    f"the farm gets **{100 - s.worst_month:.0f} %** of what it needs ({s.worst_month:.0f} % short).",
    f"Cheklashsiz xoʻjalik **100 yildan {s.years_short} tasida** taqchillik koʻradi. Oʻrtacha **yiliga "
    f"{s.mean_shortage:.1f} mln m³** yetkazib berilmaydi, bu kam. Lekin ishkal chiqqanda, qattiq chiqadi: eng yomon "
    f"oyda xoʻjalik keragining **{100 - s.worst_month:.0f} %** ini oladi ({s.worst_month:.0f} % taqchillik).",
    f"Без лимитирования хозяйство испытывает дефицит в **{s.years_short} из 100 лет**. В среднем не подаётся "
    f"**{s.mean_shortage:.1f} млн м³ в год** — немного. Но когда дело идёт плохо, оно идёт очень плохо: в худшем "
    f"месяце хозяйство получает **{100 - s.worst_month:.0f} %** потребности (дефицит {s.worst_month:.0f} %).",
)

# Второй расчёт: лимитирование

Два плана лимитирования. Оба начинают лимитировать, когда в водохранилище меньше 16 млн м³. Один урезает подачу
немного (до 80 %), другой — сильно (до 50 %).

In [ ]:
#| label: fig-second-run
#| fig-cap: "Three plans, same river. The grey line marks the trigger."
# A Plan holds the two knobs. Below 16 Mm³ in store, deliver 80 % (lightly) or 50 % (deep) of the summer need.
ration_lightly = simulate_plan(inflows, Plan(trigger=16, ration=0.8))
cut_deep = simulate_plan(inflows, Plan(trigger=16, ration=0.5))

# A dictionary of name -> run; the names become the panel titles
runs = {
    "Deliver in full until empty": no_rationing,
    "Ration lightly (below 16 Mm³, deliver 80 %)": ration_lightly,
    "Cut deep (below 16 Mm³, deliver 50 %)": cut_deep,
}
_ = plot_runs(runs, first_year=first_year)

Посмотрите на столбики справа. «Лёгкое лимитирование» превращает несколько высоких столбиков во множество низких
по 20 %, но самый высокий столбик в засуху остаётся. «Сильное урезание» ограничивает каждый столбик 50 %.

In [ ]:
#| label: tbl-three-plans
#| tbl-cap: "The three plans, scored."
#| echo: false
pd.DataFrame(
    [
        (name, run.scores.mean_shortage, run.scores.worst_month, run.scores.years_short)
        for name, run in runs.items()
    ],
    columns=["Plan", "Average shortage (Mm³/yr)", "Worst month (% short)", "Years short (of 100)"],
).style.hide(axis="index").format({"Average shortage (Mm³/yr)": "{:.1f}", "Worst month (% short)": "{:.0f}"})

In [ ]:
#| label: second-scores
#| echo: false
light, deep = ration_lightly.scores, cut_deep.scores
say(
    f"- **Rationing lightly does not help here.** The worst month is still {light.worst_month:.0f} % short: in the "
    f"worst drought the reservoir runs dry anyway. The farm is simply short more often "
    f"({light.years_short} years instead of {s.years_short}).\n"
    f"- **Cutting deep works, at a price.** The worst month improves from {s.worst_month:.0f} % to "
    f"{deep.worst_month:.0f} % short. But {deep.mean_shortage:.1f} Mm³ per year is not delivered instead of "
    f"{s.mean_shortage:.1f}, and the farm is short in {deep.years_short} years instead of {s.years_short}.\n\n"
    f"So there is no free lunch: a milder worst month costs water. How much, exactly?",
    f"- **Ozgina cheklash bu yerda yordam bermaydi.** Eng yomon oy hali ham {light.worst_month:.0f} % taqchil: eng "
    f"qattiq qurgʻoqchilikda ombor baribir quriydi. Xoʻjalik shunchaki koʻproq taqchillik koʻradi "
    f"({s.years_short} oʻrniga {light.years_short} yil).\n"
    f"- **Keskin kesish ishlaydi, lekin bahosi bor.** Eng yomon oy {s.worst_month:.0f} % dan {deep.worst_month:.0f} % "
    f"taqchillikka yaxshilanadi. Lekin {s.mean_shortage:.1f} oʻrniga yiliga {deep.mean_shortage:.1f} mln m³ "
    f"yetkazilmaydi, va xoʻjalik {s.years_short} oʻrniga {deep.years_short} yilda taqchillik koʻradi.\n\n"
    f"Demak, tekin tushlik yoʻq: yumshoqroq eng yomon oy suvga tushadi. Aniq qanchaga?",
    f"- **Лёгкое лимитирование здесь не помогает.** Худший месяц по-прежнему с дефицитом {light.worst_month:.0f} %: "
    f"в самую сильную засуху водохранилище всё равно пустеет. Хозяйство просто чаще испытывает дефицит "
    f"({light.years_short} лет вместо {s.years_short}).\n"
    f"- **Сильное урезание работает, но имеет цену.** Худший месяц улучшается с {s.worst_month:.0f} % до "
    f"{deep.worst_month:.0f} % дефицита. Но не подаётся {deep.mean_shortage:.1f} млн м³ в год вместо "
    f"{s.mean_shortage:.1f}, и хозяйство испытывает дефицит в {deep.years_short} годах вместо {s.years_short}.\n\n"
    f"Бесплатных обедов нет: более мягкий худший месяц стоит воды. Сколько именно?",
)

## Попробуйте свой план

In [ ]:
#| label: fig-my-plan
#| fig-cap: "Your plan against no rationing."
my_plan = Plan(trigger=22, ration=0.55)  # change these two numbers and run the cell again

mine = simulate_plan(inflows, my_plan)
_ = plot_runs({"Deliver in full until empty": no_rationing, "My plan": mine}, first_year=first_year)
print(mine.scores)  # the three scores of your plan

Порог может быть любым от 0 до 40 млн м³ (ёмкость), доля — любой от 0 до 1. Порог 40 означает «летом лимитировать
всегда».

# Все планы сразу

Два рычага — достаточно мало, чтобы просто перебрать всё: 21 порог × 21 доля = 441 план, 441 расчёт в TaqSim.

In [ ]:
#| label: grid
plans = plan_grid()  # 441 Plans: triggers 0, 2, 4, ... 40 Mm³ combined with rations 0, 0.05, 0.10, ... 1
table = score_plans(inflows, plans)  # runs each plan through TaqSim; one row per plan with its three scores

table.sample(5, random_state=1).sort_index().style.hide(axis="index").format(precision=2)  # five rows, as a taste

Каждая строка — один план: два рычага, три показателя и флаг `unbeaten`, объяснённый ниже. Все 441 расчёт
занимают несколько секунд.

In [ ]:
#| label: fig-maps
#| fig-cap: "Each score for every plan. Darker is worse. The three maps do not agree on where the good plans are."
#| echo: false
_ = plot_score_maps(table)

Теперь сопоставим два показателя. Каждый план становится одной точкой.

In [ ]:
#| label: fig-tradeoff
#| fig-cap: "All 441 plans. Left is better (less water undelivered). Down is better (milder worst month). A plan is 'beaten' if another plan is at least as good on both counts and better on one."
# Every row of the table becomes a dot; `named` runs are circled and labelled
_ = plot_tradeoff(table, named={"deliver in full": no_rationing, "ration lightly": ration_lightly, "cut deep": cut_deep})

Чёрные точки — **непревзойдённые планы**. Для каждого из них единственный способ смягчить худший месяц — оставить
больше воды неподанной. Только эти планы стоит обсуждать; какой из них выбрать — вопрос суждения, а не расчёта.

In [ ]:
#| label: tbl-front
#| tbl-cap: "The unbeaten plans, from least water undelivered to mildest worst month."
#| echo: false
front = front_of(table)
front[["trigger", "ration", "mean_shortage", "worst_month", "years_short"]].rename(
    columns={
        "trigger": "Trigger (Mm³)",
        "ration": "Ration",
        "mean_shortage": "Average shortage (Mm³/yr)",
        "worst_month": "Worst month (% short)",
        "years_short": "Years short (of 100)",
    }
).style.hide(axis="index").format(
    {"Trigger (Mm³)": "{:.0f}", "Ration": "{:.2f}", "Average shortage (Mm³/yr)": "{:.2f}", "Worst month (% short)": "{:.0f}"}
)

# Поручаем поиск оптимизатору

Перебор сработал, потому что рычагов всего два. У настоящего правила управления водохранилищем их десятки (свои
порог и доля для каждого месяца, несколько водохранилищ, несколько водопользователей). При десяти рычагах по 21
значению «всё» — это больше расчётов, чем кто-либо готов ждать.

Вместо перебора ищет оптимизатор (TaqSim использует метод NSGA-II). Он начинает со случайных планов, оставляет
непревзойдённые, создаёт новые планы, смешивая и слегка изменяя их, и повторяет. Мы даём ему систему и две цели:

In [ ]:
#| label: objectives-source
#| echo: false
show_source(objectives)

У каждой `Objective` есть имя, направление и функция, превращающая рассчитанную систему в одно число.

In [ ]:
#| label: optimise
found = optimise_plans(
    inflows,
    pop_size=40,  # the optimizer works with 40 plans at a time
    generations=40,  # and improves them 40 times: 1 600 TaqSim runs in all
    seed=42,  # the starting plans are random; a fixed seed makes the result reproducible
)
print(f"The optimizer returned {len(found)} plans, {len(front_of(found))} of them different.")

Внутри `optimise_plans` — один вызов TaqSim, `optimize(system, objectives, timesteps, pop_size, generations,
seed)`, после чего каждый возвращённый план пересчитывается, чтобы снять его показатели. Таблица результата имеет
те же столбцы, что и таблица перебора выше.

In [ ]:
#| label: fig-optimizer
#| fig-cap: "The optimizer's plans (blue) on top of the 441 grid plans. It finds the same curve and fills the gaps, because it is not tied to the grid's steps."
_ = plot_tradeoff(table, found=found)

# На что обратить внимание

In [ ]:
#| label: notice
#| echo: false
first, last = front.iloc[0], front.iloc[-1]
knee = front.loc[(front["worst_month"] - 50.0).abs().idxmin()]
beaten = int((~table["unbeaten"]).sum())
best_volume = table.loc[table["mean_shortage"].idxmin()]
say(
    f"**1. What you count decides the answer.** If the only goal is to deliver as much water as possible, there is "
    f"one best plan and it is *no rationing* (average shortage {best_volume['mean_shortage']:.2f} Mm³ per year, the "
    f"lowest of all 441 plans). The trade-off only appears once you also care about the worst month. Choosing the "
    f"goals is the most important modelling decision, and it is not a technical one.\n\n"
    f"**2. The curve bends.** Going from {first['worst_month']:.0f} % to {knee['worst_month']:.0f} % short in the "
    f"worst month costs {knee['mean_shortage'] - first['mean_shortage']:.1f} Mm³ per year. Going on from "
    f"{knee['worst_month']:.0f} % to {last['worst_month']:.0f} % costs another "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} Mm³ per year. The first steps are cheap, the last ones "
    f"very expensive. A plan near the bend is the natural place to start the discussion.\n\n"
    f"**3. Being short more often is part of the price.** Along the curve the number of short years rises from "
    f"{first['years_short']:.0f} to {last['years_short']:.0f} of 100.\n\n"
    f"**4. Most plans are simply bad.** {beaten} of the 441 plans are beaten, including the reasonable-sounding "
    f"\"ration lightly\". Intuition is a poor guide here; the search is not.",
    f"**1. Nimani hisoblashingiz javobni belgilaydi.** Agar yagona maqsad iloji boricha koʻp suv yetkazish boʻlsa, "
    f"bitta eng yaxshi reja bor va bu — *cheklash yoʻq* (oʻrtacha taqchillik yiliga "
    f"{best_volume['mean_shortage']:.2f} mln m³, barcha 441 reja ichida eng kami). Murosa faqat eng yomon oy ham "
    f"sizga muhim boʻlgandagina paydo boʻladi. Maqsadlarni tanlash — modellashtirishdagi eng muhim qaror, va u "
    f"texnik qaror emas.\n\n"
    f"**2. Egri chiziq bukiladi.** Eng yomon oyda {first['worst_month']:.0f} % dan {knee['worst_month']:.0f} % "
    f"taqchillikka oʻtish yiliga {knee['mean_shortage'] - first['mean_shortage']:.1f} mln m³ ga tushadi. "
    f"{knee['worst_month']:.0f} % dan {last['worst_month']:.0f} % gacha davom etish yana yiliga "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} mln m³ ga tushadi. Birinchi qadamlar arzon, oxirgilari "
    f"juda qimmat. Muhokamani bukilish yaqinidagi rejadan boshlash tabiiy.\n\n"
    f"**3. Koʻproq taqchillik koʻrish — bahoning bir qismi.** Egri chiziq boʻylab taqchil yillar soni 100 dan "
    f"{first['years_short']:.0f} tadan {last['years_short']:.0f} tagacha oshadi.\n\n"
    f"**4. Koʻpchilik rejalar shunchaki yomon.** 441 rejadan {beaten} tasi ustun kelingan, shu jumladan oqilona "
    f"tuyulgan \"ozgina cheklash\" ham. Bu yerda sezgi yomon yoʻlboshchi; qidiruv esa yaxshi.",
    f"**1. Ответ определяет то, что вы считаете.** Если единственная цель — подать как можно больше воды, есть один "
    f"лучший план, и это *без лимитирования* (средний дефицит {best_volume['mean_shortage']:.2f} млн м³ в год, "
    f"наименьший из всех 441 планов). Компромисс появляется только тогда, когда вам важен и худший месяц. Выбор "
    f"целей — самое важное решение при моделировании, и оно не техническое.\n\n"
    f"**2. Кривая изгибается.** Переход от {first['worst_month']:.0f} % к {knee['worst_month']:.0f} % дефицита в "
    f"худшем месяце стоит {knee['mean_shortage'] - first['mean_shortage']:.1f} млн м³ в год. Дальнейший переход от "
    f"{knee['worst_month']:.0f} % к {last['worst_month']:.0f} % стоит ещё "
    f"{last['mean_shortage'] - knee['mean_shortage']:.1f} млн м³ в год. Первые шаги дёшевы, последние очень дороги. "
    f"План возле изгиба — естественная отправная точка для обсуждения.\n\n"
    f"**3. Более частый дефицит — часть цены.** Вдоль кривой число дефицитных лет растёт с "
    f"{first['years_short']:.0f} до {last['years_short']:.0f} из 100.\n\n"
    f"**4. Большинство планов просто плохи.** {beaten} из 441 планов превзойдены, включая разумное на слух «лёгкое "
    f"лимитирование». Интуиция здесь плохой советчик; поиск — хороший.",
)

# Ещё один блок: испарение

До сих пор вода, сбережённая в водохранилище, ждала там бесплатно. В жарком климате так не бывает: часть её
испаряется. В TaqSim это ещё один небольшой класс, *правило потерь*, подключаемый к водохранилищу:

In [ ]:
#| label: evaporation-source
#| echo: false
show_source(SummerEvaporation)

`calculate(...)` TaqSim вызывает раз в месяц, после того как приток сохранён, и перед сбросом. Метод возвращает,
что потеряно и почему; TaqSim вычитает это из объёма и записывает событие `WaterLost`.

In [ ]:
#| label: fig-evaporation
#| fig-cap: "The unbeaten plans without and with evaporation. With evaporation every plan delivers less, and a mild worst month becomes much more expensive."
# Same 441 plans, same river, but now the reservoir loses 3 % of its content in each summer month
with_evaporation = score_plans(inflows, plan_grid(), evaporation_share=EVAPORATION_SHARE_EXAMPLE)

# Draw the unbeaten plans of both tables on one chart
_ = plot_front_shift({"no evaporation": table, "3 % of the stored water evaporates each summer month": with_evaporation})

In [ ]:
#| label: evaporation-text
#| echo: false
front_e = front_of(with_evaporation)
at_50 = front[front["worst_month"] <= 50.0].iloc[0]
at_50_e = front_e[front_e["worst_month"] <= 50.0].iloc[0]
say(
    f"The 3 % is an illustrative figure, not a measurement. The effect is large: holding the worst month to 50 % "
    f"short costs {at_50['mean_shortage']:.1f} Mm³ per year without evaporation and "
    f"{at_50_e['mean_shortage']:.1f} Mm³ per year with it. Saving water for later is worth less when the saved "
    f"water does not keep.",
    f"3 % — oʻlchov emas, tushuntirish uchun olingan raqam. Taʼsiri katta: eng yomon oyni 50 % taqchillikda "
    f"ushlab turish bugʻlanishsiz yiliga {at_50['mean_shortage']:.1f} mln m³, bugʻlanish bilan esa yiliga "
    f"{at_50_e['mean_shortage']:.1f} mln m³ ga tushadi. Saqlangan suv saqlanmasa, uni keyinga asrash kamroq "
    f"qadrli.",
    f"3 % — иллюстративное число, не измерение. Эффект велик: удержать худший месяц на уровне 50 % дефицита стоит "
    f"{at_50['mean_shortage']:.1f} млн м³ в год без испарения и {at_50_e['mean_shortage']:.1f} млн м³ в год с "
    f"ним. Беречь воду на потом менее выгодно, когда сбережённая вода не сохраняется.",
)

# Другая река: режим из статьи

Река из статьи многоводна зимой и маловодна летом — противоположность нашей. То же водохранилище, то же
хозяйство, те же сто случайных значений; только сезоны поменяны местами. Выглядит ли компромисс так же?

In [ ]:
#| label: fig-regimes
#| fig-cap: "Left: the average month of both rivers against the farm's need. Right: the unbeaten plans of both. Same reservoir, same need, same random seed."
paper_inflows = synthetic_inflows(regime=PAPER_REGIME)  # the paper's seasons: winter 40 Mm³, summer 25 Mm³
paper_table = score_plans(paper_inflows, plan_grid())  # the same 441 plans on that river

_ = plot_regime_comparison(
    {"Central Asian regime (this notebook)": (inflows, table), "Paper regime (winter peak)": (paper_inflows, paper_table)}
)

In [ ]:
#| label: regimes-text
#| echo: false
paper_front = front_of(paper_table)
paper_base = simulate_plan(paper_inflows, NO_RATIONING).scores
ours_50 = front[front["worst_month"] <= 50.0].iloc[0]
paper_50 = paper_front[paper_front["worst_month"] <= 50.0].iloc[0]
say(
    f"- **Same shape.** Both curves bend the same way: cheap at first, expensive at the end. The example works for "
    f"either river.\n"
    f"- **The paper's river is the harder case.** Without rationing it is {paper_base.worst_month:.0f} % short in "
    f"its worst month (ours: {s.worst_month:.0f} %), and holding the worst month to 50 % costs "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} Mm³ per year there against "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} here. With a summer peak the water arrives when the farm "
    f"needs it, and the reservoir only has to cover the gap.\n"
    f"- **What runs our reservoir dry** is a poor winter (little to store) followed by a weak melt. That is the "
    f"Central Asian pattern: the reservoir is filled from the low winter flow, as Kattakurgan is.",
    f"- **Shakli bir xil.** Ikkala egri chiziq ham bir xil bukiladi: avval arzon, oxirida qimmat. Misol ikkala daryo "
    f"uchun ham ishlaydi.\n"
    f"- **Maqoladagi daryo — qiyinroq holat.** Cheklashsiz uning eng yomon oyi {paper_base.worst_month:.0f} % "
    f"taqchil (bizniki: {s.worst_month:.0f} %), va eng yomon oyni 50 % da ushlab turish u yerda yiliga "
    f"{paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} mln m³, bizda esa "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} mln m³ ga tushadi. Yozgi toshqin bilan suv xoʻjalikka kerak "
    f"paytda keladi, va ombor faqat farqni qoplashi kerak.\n"
    f"- **Bizning omborni quritadigan narsa** — kam suvli qish (saqlashga kam suv) va undan keyingi kuchsiz erish. "
    f"Bu Markaziy Osiyo manzarasi: ombor, Kattaqoʻrgʻon suv ombori kabi, kam suvli qishki oqimdan toʻldiriladi.",
    f"- **Та же форма.** Обе кривые изгибаются одинаково: сначала дёшево, в конце дорого. Пример работает для обеих "
    f"рек.\n"
    f"- **Река из статьи — более трудный случай.** Без лимитирования её худший месяц имеет дефицит "
    f"{paper_base.worst_month:.0f} % (у нашей: {s.worst_month:.0f} %), а удержание худшего месяца на уровне 50 % "
    f"стоит там {paper_50['mean_shortage'] - paper_base.mean_shortage:.1f} млн м³ в год против "
    f"{ours_50['mean_shortage'] - s.mean_shortage:.1f} у нас. При летнем пике вода приходит тогда, когда она нужна "
    f"хозяйству, и водохранилищу остаётся лишь покрыть разницу.\n"
    f"- **Наше водохранилище опустошает** скудная зима (мало что накопить), за которой следует слабое таяние. Это "
    f"центральноазиатская картина: водохранилище наполняется из скудного зимнего стока, как Каттакурганское.",
)

# От игрушечной модели к настоящему водохранилищу

- **У настоящих правил те же рычаги, только много раз.** В правиле сброса Каттакурганского водохранилища в модели
  Зарафшана из этого репозитория (`ZRBReleaseRule`) есть уровень наполнения, ниже которого сброс урезается (`v1`),
  и коэффициент урезания (`buffer_coef`): наши порог и доля, с отдельным значением для каждого месяца. Поэтому
  полной модели нужен оптимизатор.
- **Чего эта игрушечная модель не учитывает.** Прогнозы предстоящего сезона; нескольких водопользователей с разными
  приоритетами; как на самом деле культуры реагируют на дефицит; испарение, зависящее от площади зеркала;
  возвратные воды.
- **Одна река — это одна река.** Числа здесь относятся к этому единственному синтетическому ряду в сто лет. Другие
  ряды дают другие числа. Форма (изогнутая кривая от «без лимитирования» до «сильного урезания») была той же для
  трёх других рядов, которые мы проверили.

# Упражнения

1. **Водохранилище побольше.** В `rationing_coursebook.py` задайте `CAPACITY_MM3 = 60`. Окупается ли
   лимитирование по-прежнему? Где теперь изгиб?
2. **Другая вторая цель.** Замените «худший месяц» на «дефицитные годы» в `objectives()` и снова запустите
   оптимизатор. Какие планы теперь непревзойдённые? Остаётся ли среди них «без лимитирования»?
3. **Другая река.** Используйте `synthetic_inflows(seed=1)`, затем `synthetic_inflows(regime=PAPER_REGIME, seed=1)`.
   Какие числа меняются, а что остаётся прежним?
4. **Сезонное правило.** Дайте правилу две доли: одну для начала лета, другую для его конца. Найдёт ли оптимизатор
   планы, превосходящие кривую с двумя рычагами?

# Источники

- Hashimoto, T., Stedinger, J. R., Loucks, D. P. (1982). Reliability, resiliency, and vulnerability criteria for
  water resource system performance evaluation. *Water Resources Research* 18(1), 14–20. The reservoir, its
  numbers (p. 17, Table 1; seasons swapped here) and the trade-off between failing often and failing badly
  (Fig. 9).
- Draper, A. J., Lund, J. R. (2004). Optimal hedging and carryover storage value. *Journal of Water Resources
  Planning and Management* 130(1), 83–87. When rationing pays (p. 83) and why a rule with two parameters is often
  enough (p. 87).
- Deb, K., Pratap, A., Agarwal, S., Meyarivan, T. (2002). A fast and elitist multiobjective genetic algorithm:
  NSGA-II. *IEEE Transactions on Evolutionary Computation* 6(2), 182–197. The search method behind
  `taqsim.optimize`.